# Abalone: Final Three Tree-Based Models (PCA)


## 1. Imports and configuration

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
)
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, RandomizedSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42
DATA_PATH = Path("data/abalone_cleaned.csv")
TARGET = "Rings"
NUMERIC_FEATURES = [
    "Length", "Diameter", "Height", "Whole weight",
    "Shucked weight", "Viscera weight", "Shell weight",
]


## 2. Read the cleaned dataset

In [2]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
            f"Could not find {DATA_PATH.resolve()}. Put abalone_cleaned.csv beside this notebook "
            "or update DATA_PATH to the correct location."
        )

df = pd.read_csv(DATA_PATH)
print("Dataset shape:", df.shape)
print("Missing values:", int(df.isna().sum().sum()))
print(df.head())

X = df.drop(columns=[TARGET])
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)
print("Train:", X_train.shape[0], "Test:", X_test.shape[0])


Dataset shape: (4024, 9)
Missing values: 0
  Sex  Length  Diameter  Height  Whole weight  Shucked weight  Viscera weight  \
0   M   0.455     0.365   0.095        0.5140          0.2245          0.1010   
1   M   0.350     0.265   0.090        0.2255          0.0995          0.0485   
2   F   0.530     0.420   0.135        0.6770          0.2565          0.1415   
3   M   0.440     0.365   0.125        0.5160          0.2155          0.1140   
4   I   0.330     0.255   0.080        0.2050          0.0895          0.0395   

   Shell weight  Rings  
0         0.150     15  
1         0.070      7  
2         0.210      9  
3         0.155     10  
4         0.055      7  
Train: 3219 Test: 805


## 3. PCA preprocessing (use only PC1 and PC2)


In [3]:
preprocessor = ColumnTransformer([
    ("numeric", "passthrough", NUMERIC_FEATURES),
    ("sex", OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False), ["Sex"]),
])

def make_pca_model(estimator):
    return Pipeline([
        ("preprocessing", clone(preprocessor)),
        ("scaler", StandardScaler()),
        ("pca", PCA(n_components=2)),
        ("model", estimator),
    ])


## 4. Three tree-based models

In [4]:
models = {
    "Random Forest": RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=1),
    "Extra Trees": ExtraTreesRegressor(random_state=RANDOM_STATE, n_jobs=1),
    "XGBoost": XGBRegressor(random_state=RANDOM_STATE, n_jobs=1, objective="reg:squarederror"),
}

param_distributions = {
    "Random Forest": {
        "model__n_estimators": [100, 200],
        "model__max_depth": [None, 10, 20],
        "model__min_samples_leaf": [1, 2, 4],
        "model__max_features": [0.7, 1.0],
    },
    "Extra Trees": {
        "model__n_estimators": [100, 200],
        "model__max_depth": [None, 10, 20],
        "model__min_samples_leaf": [1, 2, 4],
        "model__max_features": [0.7, 1.0],
    },
    "XGBoost": {
        "model__learning_rate": [0.03, 0.1],
        "model__n_estimators": [100, 200],
        "model__max_depth": [3, 5],
        "model__subsample": [0.8, 1.0],
        "model__colsample_bytree": [0.8, 1.0],
    },
}


## 5. Five-fold CV and hyperparameter tuning


In [5]:
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

best_models = {}
cv_rows = []

for name, estimator in models.items():
    search = RandomizedSearchCV(
        estimator=make_pca_model(estimator),
        param_distributions=param_distributions[name],
        n_iter=6,
        cv=cv,
        scoring="neg_mean_absolute_error",
        n_jobs=2,
        random_state=RANDOM_STATE,
        refit=True,
    )
    search.fit(X_train, y_train)

    best_models[name] = search.best_estimator_
    cv_rows.append({
        "Model": name,
        "CV_MAE": -search.best_score_,
        "Best_Parameters": str(search.best_params_),
    })
    print(name, "CV MAE:", round(-search.best_score_, 4))

cv_results = pd.DataFrame(cv_rows).sort_values("CV_MAE")
print("\nCross-validation results:")
print(cv_results.to_string(index=False))


Random Forest CV MAE: 1.841
Extra Trees CV MAE: 1.8037
XGBoost CV MAE: 1.8012

Cross-validation results:
        Model   CV_MAE                                                                                                                            Best_Parameters
      XGBoost 1.801239 {'model__subsample': 1.0, 'model__n_estimators': 100, 'model__max_depth': 3, 'model__learning_rate': 0.03, 'model__colsample_bytree': 1.0}
  Extra Trees 1.803671                             {'model__n_estimators': 200, 'model__min_samples_leaf': 1, 'model__max_features': 0.7, 'model__max_depth': 10}
Random Forest 1.841018                             {'model__n_estimators': 100, 'model__min_samples_leaf': 4, 'model__max_features': 0.7, 'model__max_depth': 10}


### 5.1 Linear Regression baseline (same PCA/data split)


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score

# Use same preprocessing + PC1/PC2 
linear_reg = make_pca_model(LinearRegression())

# five-fold CV
linear_cv_mae = -cross_val_score(
    linear_reg,
    X_train,
    y_train,
    cv=cv,
    scoring="neg_mean_absolute_error",
    n_jobs=2,
)

# Fit on the full training set
linear_reg.fit(X_train, y_train)
best_models["Linear Regression"] = linear_reg

cv_rows.append({
    "Model": "Linear Regression",
    "CV_MAE": linear_cv_mae.mean(),
    "Best_Parameters": "N/A (no tuning)",
})
cv_results = pd.DataFrame(cv_rows).sort_values("CV_MAE")

print("Linear Regression CV MAE:", round(linear_cv_mae.mean(), 4))
print("\nUpdated cross-validation results:")
print(cv_results.to_string(index=False))


Linear Regression CV MAE: 1.8708

Updated cross-validation results:
            Model   CV_MAE                                                                                                                            Best_Parameters
          XGBoost 1.801239 {'model__subsample': 1.0, 'model__n_estimators': 100, 'model__max_depth': 3, 'model__learning_rate': 0.03, 'model__colsample_bytree': 1.0}
      Extra Trees 1.803671                             {'model__n_estimators': 200, 'model__min_samples_leaf': 1, 'model__max_features': 0.7, 'model__max_depth': 10}
    Random Forest 1.841018                             {'model__n_estimators': 100, 'model__min_samples_leaf': 4, 'model__max_features': 0.7, 'model__max_depth': 10}
Linear Regression 1.870792                                                                                                                            N/A (no tuning)


## 6. Final test evaluation


In [7]:
test_rows = []
predictions = pd.DataFrame({
    "original_row_index": X_test.index,
    "actual_rings": y_test.to_numpy(),
})

for name, model in best_models.items():
    y_pred = model.predict(X_test)
    predictions[name + "_predicted_rings"] = y_pred
    test_rows.append({
        "Model": name,
        "MAE": mean_absolute_error(y_test, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_test, y_pred)),
        "R2": r2_score(y_test, y_pred),
    })

test_results = pd.DataFrame(test_rows).sort_values("MAE")
print("Test results (lower MAE/RMSE is better; higher R2 is better):")
print(test_results.round(4).to_string(index=False))


Test results (lower MAE/RMSE is better; higher R2 is better):
            Model    MAE   RMSE     R2
      Extra Trees 1.8924 2.6947 0.3078
          XGBoost 1.8925 2.6785 0.3161
    Random Forest 1.9194 2.7262 0.2915
Linear Regression 1.9644 2.7068 0.3016


## 7. PCA vs. No PCA

In [8]:
# The only change relative to make_pca_model is removal of the PCA step.
def make_no_pca_model(estimator):
    return Pipeline([
        ("preprocessing", clone(preprocessor)),
        ("scaler", StandardScaler()),
        ("model", estimator),
    ])

no_pca_best_models = {}
no_pca_cv_rows = []

# Identical 5-fold CV and RandomizedSearchCV settings as the original experiment.
for name, estimator in models.items():
    search = RandomizedSearchCV(
        estimator=make_no_pca_model(clone(estimator)),
        param_distributions=param_distributions[name],
        n_iter=6,
        cv=cv,
        scoring="neg_mean_absolute_error",
        n_jobs=2,
        random_state=RANDOM_STATE,
        refit=True,
    )
    search.fit(X_train, y_train)
    no_pca_best_models[name] = search.best_estimator_
    no_pca_cv_rows.append({
        "Model": name,
        "CV_MAE": -search.best_score_,
        "Best_Parameters": str(search.best_params_),
    })
    print("No PCA", name, "CV MAE:", round(-search.best_score_, 4))

# Same linear baseline, but trained on the original encoded features, without PCA.
no_pca_linear = make_no_pca_model(LinearRegression())
no_pca_linear_fold_mae = -cross_val_score(
    no_pca_linear, X_train, y_train,
    cv=cv, scoring="neg_mean_absolute_error", n_jobs=2,
)
no_pca_linear.fit(X_train, y_train)
no_pca_best_models["Linear Regression"] = no_pca_linear
no_pca_cv_rows.append({
    "Model": "Linear Regression",
    "CV_MAE": no_pca_linear_fold_mae.mean(),
    "Best_Parameters": "N/A (no tuning)",
})

no_pca_cv_results = pd.DataFrame(no_pca_cv_rows).sort_values("CV_MAE")

no_pca_predictions = pd.DataFrame({
    "original_row_index": X_test.index.to_numpy(),
    "actual_rings": y_test.to_numpy(),
})
no_pca_test_rows = []
for name, model in no_pca_best_models.items():
    y_pred = model.predict(X_test)
    no_pca_predictions[name + "_predicted_rings"] = y_pred
    no_pca_test_rows.append({
        "Model": name,
        "MAE": mean_absolute_error(y_test, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_test, y_pred)),
        "R2": r2_score(y_test, y_pred),
    })

no_pca_test_results = pd.DataFrame(no_pca_test_rows)

# Combine the original PCA metrics with the new No PCA metrics.
pca_vs_no_pca_test = pd.concat([
    test_results.assign(Feature_Setting="PCA (2 PCs)"),
    no_pca_test_results.assign(Feature_Setting="No PCA (9 encoded features)"),
], ignore_index=True)[["Feature_Setting", "Model", "MAE", "RMSE", "R2"]]

pca_vs_no_pca_cv = pd.concat([
    cv_results.assign(Feature_Setting="PCA (2 PCs)"),
    no_pca_cv_results.assign(Feature_Setting="No PCA (9 encoded features)"),
], ignore_index=True)[["Feature_Setting", "Model", "CV_MAE", "Best_Parameters"]]

print("\nPCA vs No PCA on the SAME held-out test set:")
print(pca_vs_no_pca_test.sort_values(["Model", "Feature_Setting"]).round(4).to_string(index=False))

No PCA Random Forest CV MAE: 1.4917
No PCA Extra Trees CV MAE: 1.5055
No PCA XGBoost CV MAE: 1.5062

PCA vs No PCA on the SAME held-out test set:
            Feature_Setting             Model    MAE   RMSE     R2
No PCA (9 encoded features)       Extra Trees 1.5763 2.2263 0.5275
                PCA (2 PCs)       Extra Trees 1.8924 2.6947 0.3078
No PCA (9 encoded features) Linear Regression 1.6427 2.2398 0.5218
                PCA (2 PCs) Linear Regression 1.9644 2.7068 0.3016
No PCA (9 encoded features)     Random Forest 1.5579 2.1990 0.5391
                PCA (2 PCs)     Random Forest 1.9194 2.7262 0.2915
No PCA (9 encoded features)           XGBoost 1.5636 2.2149 0.5324
                PCA (2 PCs)           XGBoost 1.8925 2.6785 0.3161


## 8. web demo part


In [9]:
# Align feature values using the original test indices (never merge by row position).
interactive_predictions = predictions.copy()
for col in X_test.columns:
    interactive_predictions[col] = X_test.loc[interactive_predictions["original_row_index"], col].to_numpy()

model_names = list(best_models.keys())
actual = interactive_predictions["actual_rings"].to_numpy()

for name in model_names:
    pca_col = name + "_predicted_rings"  # Retain the existing column name.
    no_pca_col = "NoPCA_" + name + "_predicted_rings"
    interactive_predictions[no_pca_col] = no_pca_predictions[name + "_predicted_rings"].to_numpy()
    interactive_predictions["PCA_" + name + "_absolute_error"] = np.abs(
        interactive_predictions[pca_col].to_numpy() - actual
    )
    interactive_predictions["NoPCA_" + name + "_absolute_error"] = np.abs(
        interactive_predictions[no_pca_col].to_numpy() - actual
    )

# Optional row-level winner labels for the site (ties default to first model in model_names).
for label in ["PCA", "NoPCA"]:
    errors = interactive_predictions[[
        label + "_" + name + "_absolute_error" for name in model_names
    ]].to_numpy()
    interactive_predictions[label + "_best_model_for_row"] = np.array(model_names)[
        np.argmin(errors, axis=1)
    ]

# Convenient column order for JavaScript and charts.
first_cols = ["original_row_index", "actual_rings"] + list(X_test.columns)
interactive_predictions = interactive_predictions[
    first_cols + [c for c in interactive_predictions if c not in first_cols]
]

print("Interactive prediction dataset:", interactive_predictions.shape)
print(interactive_predictions.head(3).to_string(index=False))

Interactive prediction dataset: (805, 28)
 original_row_index  actual_rings Sex  Length  Diameter  Height  Whole weight  Shucked weight  Viscera weight  Shell weight  Random Forest_predicted_rings  Extra Trees_predicted_rings  XGBoost_predicted_rings  Linear Regression_predicted_rings  NoPCA_Random Forest_predicted_rings  PCA_Random Forest_absolute_error  NoPCA_Random Forest_absolute_error  NoPCA_Extra Trees_predicted_rings  PCA_Extra Trees_absolute_error  NoPCA_Extra Trees_absolute_error  NoPCA_XGBoost_predicted_rings  PCA_XGBoost_absolute_error  NoPCA_XGBoost_absolute_error  NoPCA_Linear Regression_predicted_rings  PCA_Linear Regression_absolute_error  NoPCA_Linear Regression_absolute_error PCA_best_model_for_row NoPCA_best_model_for_row
               1718             9   M    0.57     0.455   0.175         1.020          0.4805          0.2145        0.2900                      11.544177                    11.350523                11.129944                          11.139485       

## 9. Paired bootstrap: 95% confidence intervals for MAE differences

In [10]:
from itertools import combinations

rng = np.random.default_rng(RANDOM_STATE)
N_BOOTSTRAP = 3000
row_indices = rng.integers(
    0, len(interactive_predictions),
    size=(N_BOOTSTRAP, len(interactive_predictions)),
)

# Prepare one paired bootstrap distribution of MAE for each setting/model.
errors_by_model = {}
bootstrap_mae = {}
for setting in ["PCA", "NoPCA"]:
    for name in model_names:
        key = (setting, name)
        err = interactive_predictions[
            setting + "_" + name + "_absolute_error"
        ].to_numpy()
        errors_by_model[key] = err
        bootstrap_mae[key] = err[row_indices].mean(axis=1)

model_mae_ci_rows = []
for (setting, name), err in errors_by_model.items():
    low, high = np.quantile(bootstrap_mae[(setting, name)], [0.025, 0.975])
    model_mae_ci_rows.append({
        "Feature_Setting": setting,
        "Model": name,
        "Test_MAE": err.mean(),
        "MAE_CI_Lower_95": low,
        "MAE_CI_Upper_95": high,
        "Test_Samples": len(err),
        "Bootstrap_Resamples": N_BOOTSTRAP,
    })
bootstrap_model_mae_ci = pd.DataFrame(model_mae_ci_rows)

comparison_pairs = []
for setting in ["PCA", "NoPCA"]:
    for model_a, model_b in combinations(model_names, 2):
        comparison_pairs.append(((setting, model_a), (setting, model_b)))
for name in model_names:
    comparison_pairs.append((("PCA", name), ("NoPCA", name)))

paired_mae_rows = []
for key_a, key_b in comparison_pairs:
    diff_samples = bootstrap_mae[key_a] - bootstrap_mae[key_b]
    ci_low, ci_high = np.quantile(diff_samples, [0.025, 0.975])
    observed = errors_by_model[key_a].mean() - errors_by_model[key_b].mean()
    paired_mae_rows.append({
        "Setting_A": key_a[0],
        "Model_A": key_a[1],
        "Setting_B": key_b[0],
        "Model_B": key_b[1],
        "MAE_Difference_A_minus_B": observed,
        "CI_Lower_95": ci_low,
        "CI_Upper_95": ci_high,
        "CI_Excludes_Zero": bool(ci_low > 0 or ci_high < 0),
        "Test_Samples": len(interactive_predictions),
        "Bootstrap_Resamples": N_BOOTSTRAP,
    })
paired_bootstrap_mae = pd.DataFrame(paired_mae_rows)

print("\nPaired MAE differences (negative = Model A wins):")
print(paired_bootstrap_mae.round(4).to_string(index=False))


Paired MAE differences (negative = Model A wins):
Setting_A           Model_A Setting_B           Model_B  MAE_Difference_A_minus_B  CI_Lower_95  CI_Upper_95  CI_Excludes_Zero  Test_Samples  Bootstrap_Resamples
      PCA     Random Forest       PCA       Extra Trees                    0.0270       0.0041       0.0516              True           805                 3000
      PCA     Random Forest       PCA           XGBoost                    0.0269      -0.0073       0.0602             False           805                 3000
      PCA     Random Forest       PCA Linear Regression                   -0.0449      -0.0993       0.0100             False           805                 3000
      PCA       Extra Trees       PCA           XGBoost                   -0.0002      -0.0195       0.0203             False           805                 3000
      PCA       Extra Trees       PCA Linear Regression                   -0.0720      -0.1178      -0.0254              True           805     

## 10. Export all results

All CSV exports are centralized here, **after** every training, evaluation, PCA comparison, prediction, and bootstrap step has finished.

Eight CSV files are retained because metrics, per-row predictions, and bootstrap comparisons have different structures. Keeping their original filenames preserves compatibility with the existing static web demo and teammates' scripts.

Run this section once after the earlier cells, rather than exporting multiple times during training.

In [11]:
results_dir = Path("results")
results_dir.mkdir(parents=True, exist_ok=True)

# Export everything in one place; keep filenames unchanged for the web demo.
export_files = {
    # Original PCA-only results (all four models).
    "pca_three_trees_cv.csv": cv_results,
    "pca_three_trees_test_metrics.csv": test_results,
    "pca_three_trees_predictions.csv": predictions,
    # PCA vs. No PCA and interactive web results.
    "pca_vs_no_pca_test_metrics.csv": pca_vs_no_pca_test,
    "pca_vs_no_pca_cv.csv": pca_vs_no_pca_cv,
    "interactive_predictions.csv": interactive_predictions,
    # Bootstrap uncertainty estimates.
    "bootstrap_model_mae_ci.csv": bootstrap_model_mae_ci,
    "paired_bootstrap_mae.csv": paired_bootstrap_mae,
}

for filename, frame in export_files.items():
    file_path = results_dir / filename
    frame.to_csv(file_path, index=False)
    print("Saved:", file_path, "|", len(frame), "rows")


Saved: results\pca_three_trees_cv.csv | 4 rows
Saved: results\pca_three_trees_test_metrics.csv | 4 rows
Saved: results\pca_three_trees_predictions.csv | 805 rows
Saved: results\pca_vs_no_pca_test_metrics.csv | 8 rows
Saved: results\pca_vs_no_pca_cv.csv | 8 rows
Saved: results\interactive_predictions.csv | 805 rows
Saved: results\bootstrap_model_mae_ci.csv | 8 rows
Saved: results\paired_bootstrap_mae.csv | 16 rows
